# Bài 43: Phân đoạn thực thể

Phân đoạn ngữ nghĩa (Bài 42) gán nhãn mỗi điểm ảnh với một lớp, nhưng nó không có khái niệm về việc mỗi lớp có *bao nhiêu* thực thể — hai thực thể chạm nhau chỉ được gán nhãn là một khối điểm ảnh liên thông. **Phân đoạn thực thể (instance segmentation)** trả lời câu hỏi khó hơn: những điểm ảnh nào thuộc về đối tượng *cụ thể này*, khác với một đối tượng nào đó khác cùng lớp. Bài này mở rộng trực tiếp mạng phân đoạn ở Bài 42 để giải quyết nó, với một ý tưởng truy thẳng về biến đổi Hough ở Bài 12: thay vì chỉ phân loại điểm ảnh, cho mạng dự đoán thêm, với mỗi điểm ảnh tiền cảnh, một phiếu bầu về vị trí tâm của đối tượng chứa nó — rồi phân cụm các phiếu để khôi phục từng thực thể riêng lẻ.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['apollo11_crew.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Các thực thể chồng lấn

Hai hình tròn mỗi cảnh, được cố tình đặt đủ gần để chạm hoặc chồng lên nhau. Dữ liệu chuẩn bao gồm cả mặt nạ ngữ nghĩa (0=nền, 1=hình tròn) lẫn mặt nạ *thực thể* (0=nền, 1=hình tròn thứ nhất, 2=hình tròn thứ hai).

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, n_circles=2, r=5):
    scene = np.zeros((size, size), dtype=np.float32)
    sem_mask = np.zeros((size, size), dtype=np.int64)
    inst_mask = np.zeros((size, size), dtype=np.int64)
    # Both circles are the same semantic class, so the two instances are
    # distinguished only by draw order (k=0 vs k=1), not by any fixed
    # category. Using different fixed intensities per instance index means
    # whichever circle is drawn last "wins" identically in BOTH scene and
    # inst_mask at the overlap -- so the pixel's intensity always matches its
    # instance id there, instead of being visually identical either way.
    intensity = {0: 0.85, 1: 1.0}
    centers = []
    for k in range(n_circles):
        if k == 0:
            cx, cy = rng.integers(r + 8, size - r - 8), rng.integers(r + 8, size - r - 8)
        else:
            px, py = centers[-1]
            angle = rng.uniform(0, 2 * np.pi)
            dist = rng.uniform(6, 9)  # overlapping (< 2r) but centers still separable
            cx = int(np.clip(px + dist * np.cos(angle), r, size - r - 1))
            cy = int(np.clip(py + dist * np.sin(angle), r, size - r - 1))
        centers.append((cx, cy))
        yy, xx = np.mgrid[0:size, 0:size]
        m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        scene[m] = intensity[k]
        sem_mask[m] = 1
        inst_mask[m] = k + 1  # later circles paint over earlier ones at overlaps
    scene = np.clip(scene + rng.normal(0, 0.05, scene.shape), 0, 1).astype(np.float32)
    return scene, sem_mask, inst_mask, centers

rng = np.random.default_rng(17)
N = 300
scenes, sem_masks, inst_masks, all_centers = [], [], [], []
for _ in range(N):
    s, sm, im, c = make_scene(rng)
    scenes.append(s); sem_masks.append(sm); inst_masks.append(im); all_centers.append(c)
scenes = np.array(scenes, dtype=np.float32)
sem_masks = np.array(sem_masks, dtype=np.int64)

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for i in range(4):
    axes[0, i].imshow(scenes[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(inst_masks[i], cmap='viridis'); axes[1, i].axis('off')
axes[0, 0].set_title('input', fontsize=9, loc='left')
axes[1, 0].set_title('ground truth instance mask', fontsize=9, loc='left')
plt.show()

## Mở rộng mạng với một phần đầu thực thể

Lấy chính xác kiến trúc U-Net ở Bài 42 và cho nó một phần đầu đầu ra thứ hai bên cạnh phần đầu ngữ nghĩa: với mỗi điểm ảnh tiền cảnh, dự đoán một độ lệch `(dx, dy)` chỉ về tâm thực thể của chính nó. Phần đầu ngữ nghĩa vẫn làm cùng việc như Bài 42 (nền và hình tròn); phần đầu độ lệch mới là thứ biến phân đoạn đơn thuần thành phân đoạn *thực thể*.

In [ ]:
# build offset targets: for every foreground pixel, the (dx, dy) to ITS instance's center
yy, xx = np.mgrid[0:SIZE, 0:SIZE]
offset_targets = np.zeros((N, 2, SIZE, SIZE), dtype=np.float32)
for i in range(N):
    im = inst_masks[i]
    for k, (cx, cy) in enumerate(all_centers[i]):
        m = im == (k + 1)
        offset_targets[i, 0][m] = (cx - xx[m]) / SIZE
        offset_targets[i, 1][m] = (cy - yy[m]) / SIZE

split = int(0.85 * N)
Xtr, Str, Otr = scenes[:split], sem_masks[:split], offset_targets[:split]
Xte, Ste, Ote = scenes[split:], sem_masks[split:], offset_targets[split:]
inst_te, centers_te = inst_masks[split:], all_centers[split:]

class InstanceNet(nn.Module):
    """Lesson 42's U-Net, with a second head: per-pixel offset-to-center regression"""
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU())
        self.sem_head = nn.Conv2d(16, 2, 1)     # background vs. circle
        self.offset_head = nn.Conv2d(16, 2, 1)  # (dx, dy) to this pixel's instance center

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool(f1))
        d1 = self.dec1(torch.cat([self.up(f2), f1], dim=1))
        return self.sem_head(d1), self.offset_head(d1)

torch.manual_seed(0)
model = InstanceNet()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); St = torch.tensor(Str); Ot = torch.tensor(Otr)
fg_mask_t = (St == 1).unsqueeze(1).float()

for _ in range(300):
    opt.zero_grad()
    sem_logits, offset_pred = model(Xt)
    sem_loss = F.cross_entropy(sem_logits, St)
    offset_loss = (F.mse_loss(offset_pred, Ot, reduction='none') * fg_mask_t).sum() / fg_mask_t.sum().clamp(min=1)
    (sem_loss + 2.0 * offset_loss).backward()
    opt.step()

with torch.no_grad():
    sem_logits_te, offset_pred_te = model(torch.tensor(Xte).unsqueeze(1))
    sem_preds = sem_logits_te.argmax(1).numpy()
    offset_preds = offset_pred_te.numpy()

print(f'semantic pixel accuracy: {(sem_preds == Ste).mean():.1%}')


## Khôi phục thực thể: bỏ phiếu cho tâm, rồi phân cụm (lại là Hough)

Phần đầu độ lệch ở trên được huấn luyện để dự đoán, với mỗi điểm ảnh tiền cảnh, một độ lệch chỉ về tâm *thực thể của chính nó* — được giám sát trực tiếp từ các mặt nạ thực thể chuẩn. Mỗi điểm ảnh tiền cảnh bỏ một phiếu (vị trí tâm dự đoán của nó) vào một bộ tích lũy, y hệt như bỏ phiếu đường thẳng/đường tròn Hough ở Bài 12. Vì mọi điểm ảnh thuộc cùng một hình tròn đều bỏ phiếu cho xấp xỉ cùng một điểm, bộ tích lũy hình thành một cụm phiếu chặt cho mỗi thực thể, bất kể bản thân các điểm ảnh đan xen thế nào. Tìm thực thể trở thành: tìm các cụm phiếu, rồi gán mỗi điểm ảnh cho cụm gần nhất.

In [ ]:
def cluster_instances(binary_mask, offset_pred, size=SIZE, peak_dist=4, vote_thresh=3):
    ys, xs = np.where(binary_mask)
    if len(xs) == 0:
        return np.zeros_like(binary_mask, dtype=np.int64), []
    pred_cx = xs + offset_pred[0][ys, xs] * size
    pred_cy = ys + offset_pred[1][ys, xs] * size

    votes = np.zeros((size, size), dtype=np.float32)
    for cx, cy in zip(pred_cx, pred_cy):
        cxi = int(np.clip(round(cx), 0, size - 1)); cyi = int(np.clip(round(cy), 0, size - 1))
        votes[cyi, cxi] += 1

    # greedily take the highest-voted peak, suppress its neighborhood, repeat
    # (the same non-max-suppression idea as Lesson 40's detection boxes)
    peaks = []
    votes_work = votes.copy()
    for _ in range(6):
        idx = np.unravel_index(votes_work.argmax(), votes_work.shape)
        if votes_work[idx] < vote_thresh:
            break
        peaks.append((idx[1], idx[0]))
        y0, y1 = max(0, idx[0] - peak_dist), min(size, idx[0] + peak_dist + 1)
        x0, x1 = max(0, idx[1] - peak_dist), min(size, idx[1] + peak_dist + 1)
        votes_work[y0:y1, x0:x1] = 0

    if not peaks:
        return np.zeros_like(binary_mask, dtype=np.int64), []
    inst_pred = np.zeros_like(binary_mask, dtype=np.int64)
    peaks_arr = np.array(peaks)
    for y, x in zip(ys, xs):
        dists = (peaks_arr[:, 0] - x) ** 2 + (peaks_arr[:, 1] - y) ** 2
        inst_pred[y, x] = np.argmin(dists) + 1
    return inst_pred, peaks

n_correct = 0
inst_preds, all_peaks = [], []
for i in range(len(Xte)):
    binary_mask = sem_preds[i] == 1
    inst_pred, peaks = cluster_instances(binary_mask, offset_preds[i])
    inst_preds.append(inst_pred); all_peaks.append(peaks)
    true_n = len(set(inst_te[i].ravel()) - {0})
    if len(peaks) == true_n:
        n_correct += 1

print(f'offset-voting recovers the correct instance count: {n_correct}/{len(Xte)} scenes '
      f'({n_correct/len(Xte):.1%})')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 6.5))
for i in range(4):
    axes[0, i].imshow(scenes[split + i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(inst_te[i], cmap='viridis'); axes[1, i].axis('off')
    axes[2, i].imshow(inst_preds[i], cmap='viridis'); axes[2, i].axis('off')
for r, name in enumerate(['input', 'ground truth instances', 'recovered via offset voting']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

Có hai điểm đáng chú ý từ hình. Thứ nhất, bản thân các màu là tùy ý: `cluster_instances` đánh số các đỉnh theo thứ tự nó *tìm thấy* chúng (đỉnh nhiều phiếu nhất trước), không liên quan gì đến việc `make_scene` tình cờ vẽ hình tròn nào trước. Danh tính thực thể ở đây là tùy ý, giống như một nhãn thành phần liên thông: điều quan trọng là *tập* điểm ảnh được gán cho mỗi thực thể có đúng không, không phải màu nào được gán cho tập nào. Đây là lý do, ở ba trong bốn cột, "thực thể 1" (xanh mòng két) và "thực thể 2" (vàng) bị hoán đổi giữa hàng dữ liệu chuẩn và hàng khôi phục được. Đó không phải là lỗi.

Thứ hai, cột thứ hai *thực sự* cho thấy một lỗi, vì bước phân cụm không tách được hai hình tròn &mdash; thay vào đó khôi phục một khối đặc duy nhất. Đó là vì hai tâm hình tròn của cảnh này chỉ cách nhau 5.0 điểm ảnh (so với 7.6-8.6 ở ba cột còn lại), gần hơn khoảng `dist` danh nghĩa 6-9 của vòng lặp do việc kẹp biên trong `make_scene`. Lần theo `cluster_instances` trên cảnh này: nó chọn đỉnh đầu tiên với 25 phiếu, rồi triệt tiêu một vùng lân cận 9x9 quanh đó (`peak_dist=4` theo cả hai chiều) để tìm đỉnh tiếp theo — nhưng đỉnh phiếu thực của hình tròn thứ hai rơi *bên trong* cửa sổ triệt tiêu đó, nên nó bị đưa về 0 trước khi vòng lặp kịp tới nó. Chỉ một đỉnh sống sót, nên cả hai hình tròn sụp đổ thành một thực thể duy nhất.

Ảnh động này chạy quá trình đó trên ảnh đầu tiên hiển thị ở trên: các điểm ảnh tiền cảnh bỏ phiếu vào bộ tích lũy, hai cụm chặt hiện ra, và cùng vòng lặp chọn-đỉnh/triệt-tiêu tham lam ở trên khôi phục hai tâm thực thể, kết thúc với cùng phép gán thực thể hiển thị ở trên.

<img src="../animations/instance_voting_demo.gif" alt="Ảnh động khôi phục thực thể bằng bỏ phiếu tâm: các điểm ảnh tiền cảnh từ hai hình tròn chạm nhau bỏ phiếu tích tụ thành hai cụm chặt trong một bộ tích lũy, rồi một vòng lặp chọn-đỉnh-và-triệt-tiêu tham lam (khớp với NMS ở Bài 40) tìm ra cả hai tâm cụm và gán mọi điểm ảnh tiền cảnh cho tâm gần nhất, khớp chính xác với mặt nạ hai thực thể chuẩn" style="display: block; margin: 0 auto; max-width: 760px;">

## Mask R-CNN: Faster R-CNN cộng một nhánh mặt nạ

Cách tiếp cận bỏ phiếu độ lệch ở trên là một họ phương pháp phân đoạn thực thể (liên quan đến các kỹ thuật đôi khi gọi là "nhúng thực thể (instance embedding)" hoặc bỏ phiếu tâm). Cách tiếp cận thống trị còn lại, **Mask R-CNN** (<a href="../references.html#he-2017-maskrcnn">He et al., 2017</a><span class="landmark-paper">&#9733;</span>), không bắt đầu từ đầu — nó là **Faster R-CNN** (Bài 41) với một thành phần mới được gắn thêm. Mạng đề xuất vùng, bộ phân loại theo từng hộp và phép tinh chỉnh tọa độ theo từng hộp đều giữ nguyên. Mask R-CNN chỉ thêm một nhánh thứ ba, song song trên mỗi hộp được phát hiện: cạnh nhãn lớp và tọa độ hộp đã tinh chỉnh sẵn có, một phần đầu tích chập hoàn toàn nhỏ dự đoán một mặt nạ nhị phân *bên trong hộp đó*. Phát hiện đã giải quyết "có bao nhiêu đối tượng, và ở đại khái đâu" qua đề xuất vùng và NMS; nhánh mặt nạ thêm vào "và đây là hình bóng chính xác của đối tượng này" — chính là thành phần được xây dựng và kiểm tra riêng lẻ bên dưới.

Cần thêm một thay đổi nữa để nhánh mặt nạ đó thực sự hoạt động tốt. Faster R-CNN trích xuất một lưới đặc trưng kích thước cố định cho mỗi hộp đề xuất bằng cách bám (lượng tử hóa) tọa độ của hộp vào ô bản đồ đặc trưng gần nhất — một bước gọi là RoIPool. Phép làm tròn đó hầu như không ảnh hưởng đến việc phân loại một hộp hay xê dịch tọa độ của nó vài điểm ảnh, nhưng nó làm lệch thấy rõ một mặt nạ theo từng điểm ảnh so với đối tượng mà nó phải viền quanh. Mask R-CNN thay RoIPool bằng **RoIAlign**, đọc đặc trưng qua nội suy song tuyến tính thay vì làm tròn về ô gần nhất, để các đặc trưng trích xuất được — và do đó mặt nạ dự đoán — luôn thẳng hàng với tọa độ hộp thực tế. Đây là một thay đổi nhỏ đối với bộ phát hiện, nhưng nó là cách khắc phục cụ thể giúp tạo ra được các mặt nạ chính xác đến từng điểm ảnh.

Ghép Bài 36, 40, 41 và bài này lại: **phân loại** (Bài 36) cho một nhãn cho cả ảnh, **phân đoạn ngữ nghĩa** (Bài 42) cho một nhãn cho mỗi điểm ảnh mà không có khái niệm về các đối tượng riêng biệt cùng lớp, và **phân đoạn thực thể** (bài này) thêm một danh tính riêng biệt cho mỗi đối tượng — "nền", "hình tròn #1", "hình tròn #2", không chỉ "nền", "hình tròn". Một thuật ngữ thứ tư, **phân đoạn toàn cảnh (panoptic segmentation)**, hợp nhất hai loại sau: mọi điểm ảnh nhận cả một lớp ngữ nghĩa và, với các lớp "vật (thing)" đếm được như hình tròn hay ô tô, một ID thực thể — trong khi các lớp "chất (stuff)" không đếm được như bầu trời hay mặt đường chỉ được gán nhãn ngữ nghĩa, vì danh tính thực thể không có ý nghĩa với chúng.

Ảnh động bên dưới minh họa sự lệch trên một lưới đặc trưng nhỏ: hãy quan sát vùng trích xuất bám theo một hộp đang trượt một cách mượt mà với phép lấy mẫu song tuyến tính của RoIAlign, nhưng nhảy theo các bước rời rạc với phép làm tròn của RoIPool.

<img src="../animations/roi_align_demo.gif" alt="Ảnh động so sánh RoIPool và RoIAlign trên một lưới đặc trưng nhỏ 8x8: khi một hộp đề xuất trượt qua các vị trí dưới điểm ảnh, vùng trích xuất đã lượng tử hóa của RoIPool (cam) nhảy giữa các ô lưới và lệch thấy rõ so với các cạnh thực của hộp, trong khi các điểm lấy mẫu song tuyến tính của RoIAlign (xanh lục) bám theo hộp một cách mượt mà và luôn nằm giữa nó ở mọi vị trí" style="display: block; margin: 0 auto; max-width: 760px;">

## Một phần đầu mặt nạ

Phát hiện (Bài 41) và tạo mặt nạ theo từng thực thể là hai thành phần riêng biệt, đã được giải quyết tại thời điểm này — thành phần thực sự mới duy nhất mà Mask R-CNN thêm vào là **phần đầu mặt nạ (mask head)**: một mạng nhỏ *chỉ* nhìn vào bên trong một hộp đề xuất và dự đoán một mặt nạ nhị phân cho đối tượng duy nhất mà hộp đó được đề xuất, bỏ qua mọi thứ khác nhìn thấy trong vùng cắt.

Hãy xây dựng thành phần đó một cách riêng lẻ. Lấy hộp chuẩn quanh tâm thực của mỗi thực thể (giả định việc định vị đã được giải quyết, để chỉ ý tưởng phần đầu mặt nạ mới là đối tượng kiểm tra), cắt một cửa sổ cố định 14x14 ở đó, và huấn luyện một phần đầu tích chập hoàn toàn nhỏ bé để dự đoán những điểm ảnh nào trong vùng cắt đó thuộc về thực thể *này* — không phải thuộc về bất kỳ hình tròn lân cận nào tình cờ cũng nhìn thấy trong cùng vùng cắt.

In [ ]:
CROP = 14
HALF = CROP // 2

def make_crops(indices):
    crops, masks = [], []
    for i in indices:
        scene, inst = scenes[i], inst_masks[i]
        for k, (cx, cy) in enumerate(all_centers[i]):
            x0 = int(np.clip(cx - HALF, 0, SIZE - CROP))
            y0 = int(np.clip(cy - HALF, 0, SIZE - CROP))
            crop = scene[y0:y0 + CROP, x0:x0 + CROP]
            inst_crop = inst[y0:y0 + CROP, x0:x0 + CROP]
            mask = (inst_crop == (k + 1)).astype(np.float32)  # only THIS instance, never the neighbor
            crops.append(crop); masks.append(mask)
    return np.array(crops, dtype=np.float32), np.array(masks, dtype=np.float32)

X_crop_train, M_crop_train = make_crops(range(split))
X_crop_test, M_crop_test = make_crops(range(split, N))
print(f'{len(X_crop_train)} training crops, {len(X_crop_test)} test crops, each {CROP}x{CROP}')

class MaskHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1),  # per-pixel mask logit, full crop resolution throughout
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
mask_head = MaskHead()
opt = torch.optim.Adam(mask_head.parameters(), lr=0.01)
Xc = torch.tensor(X_crop_train).unsqueeze(1)
Mc = torch.tensor(M_crop_train).unsqueeze(1)
for _ in range(300):
    opt.zero_grad()
    loss = F.binary_cross_entropy_with_logits(mask_head(Xc), Mc)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_masks = (mask_head(torch.tensor(X_crop_test).unsqueeze(1)) > 0).float().squeeze(1).numpy()

def mean_iou(pred, true):
    inter = (pred * true).sum(axis=(1, 2))
    union = ((pred + true) > 0).sum(axis=(1, 2))
    return (inter / np.clip(union, 1, None)).mean()

naive_masks = (X_crop_test > 0.5).astype(np.float32)  # "everything foreground in the crop", no per-instance awareness
print(f'mask-head IoU (per-instance):          {mean_iou(pred_masks, M_crop_test):.3f}')
print(f'naive foreground-in-crop baseline IoU: {mean_iou(naive_masks, M_crop_test):.3f}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(9, 5))
for i in range(4):
    axes[0, i].imshow(X_crop_test[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(M_crop_test[i], cmap='viridis', vmin=0, vmax=1); axes[1, i].axis('off')
    axes[2, i].imshow(pred_masks[i], cmap='viridis', vmin=0, vmax=1); axes[2, i].axis('off')
for r, name in enumerate(['crop (neighbor circle visible)', 'ground truth mask (this instance only)', 'predicted mask']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

Ảnh động bên dưới quan sát phần đầu mặt nạ học, trên cùng bốn vùng cắt được giữ lại này, cách tách các điểm ảnh của thực thể này khỏi hình tròn lân cận nhìn thấy được — bắt đầu từ một phỏng đoán nhiễu, gần như ngẫu nhiên và hội tụ về các mặt nạ theo từng thực thể hiển thị ở trên.

<img src="../animations/mask_head_training_demo.gif" alt="Ảnh động huấn luyện phần đầu mặt nạ: trên bốn vùng cắt được giữ lại, mỗi vùng cho thấy một hình tròn lân cận bên cạnh thực thể đích, mặt nạ dự đoán bắt đầu là nhiễu gần như ngẫu nhiên và dần hội tụ để viền chính xác chỉ thực thể đích, loại trừ đúng hình tròn lân cận nhìn thấy được, với IoU trung bình tăng dần khi huấn luyện tiến triển" style="display: block; margin: 0 auto; max-width: 820px;">

Vì các hình tròn của tập dữ liệu này được tạo ra đủ gần để luôn chạm hoặc chồng lên nhau, *mọi vùng cắt kiểm tra* đều chứa các điểm ảnh nhìn thấy được từ hình tròn lân cận — không có trường hợp "dễ" nào ở đây để thổi phồng các con số. Phần đầu mặt nạ theo từng thực thể vẫn đạt IoU cao đáng nể so với mặt nạ thực thể đơn chuẩn, vượt rõ so với đường cơ sở ngây thơ chỉ đánh dấu mọi điểm ảnh tiền cảnh trong vùng cắt là thuộc về "cái" đối tượng (vốn sai một cách có hệ thống bất cứ khi nào một hình lân cận xâm phạm — cùng vấn đề đã thúc đẩy việc cho mạng một phần đầu tâm thực thể ngay từ đầu, chỉ là diễn lại ở quy mô vùng cắt). Phần đầu mặt nạ không cần giải quyết "các hình tròn ở đâu" — hộp đã trả lời điều đó — nó chỉ phải học "biết rằng hộp này được đề xuất cho một hình tròn cụ thể, những điểm ảnh nào là của hình tròn đó, chứ không phải của hình lân cận". Đó là một câu hỏi dễ hơn hẳn, cục bộ hơn, đó chính là lý do nhánh mặt nạ của Mask R-CNN có thể nhỏ và nhanh: một hộp tốt đã làm phần lớn công việc.

Điều này cũng làm rõ tại sao Mask R-CNN và cách tiếp cận bỏ phiếu độ lệch của bài này hiếm khi thất bại theo cùng một cách. Bỏ phiếu độ lệch có thể phân cụm sai các phiếu khi tâm của hai thực thể đủ gần để các đỉnh bộ tích lũy của chúng nhòe vào nhau (Bài tập 1 bên dưới khám phá điều này). Mask R-CNN thay vào đó phụ thuộc vào việc bộ phát hiện phía trước đề xuất một hộp đúng cho mọi thực thể ngay từ đầu — bỏ sót một hộp, và mặt nạ của thực thể đó thậm chí không bao giờ được thử, bất kể phần đầu mặt nạ tốt đến đâu.

## Trong thực tế: Mask R-CNN trên một bức ảnh thật

Phần đầu mặt nạ ở trên chứng minh ý tưởng hoạt động khi việc định vị được giải quyết miễn phí (tâm hộp chuẩn). Dưới đây là cùng ý tưởng đó, được lắp ráp đầy đủ và tiền huấn luyện: một **Mask R-CNN** thật (`torchvision.models.detection.maskrcnn_resnet50_fpn`, huấn luyện trên COCO) chạy từ đầu đến cuối — bao gồm cả phát hiện lẫn phần đầu mặt nạ, không cho sẵn gì — trên một bức ảnh với đúng loại thực thể cùng lớp, chạm nhau mà bài này đã hướng tới.

In [ ]:
import torchvision
import cv2
import matplotlib.patches as mpatches

weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.COCO_V1
maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights)
maskrcnn.eval()  # frozen, no training at all
coco_classes = weights.meta['categories']

photo = cv2.imread('../img/apollo11_crew.jpg')
photo_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)

preprocess = weights.transforms()
photo_tensor = torch.tensor(photo_rgb / 255.0, dtype=torch.float32).permute(2, 0, 1)
batch = preprocess(photo_tensor).unsqueeze(0)

with torch.no_grad():
    result = maskrcnn(batch)[0]

score_thresh = 0.8
keep = result['scores'] > score_thresh
labels = result['labels'][keep].numpy()
scores = result['scores'][keep].numpy()
masks = result['masks'][keep, 0].numpy()  # soft per-instance masks

print(f'{keep.sum().item()} instances above {score_thresh:.0%} confidence:')
for lbl, sc in zip(labels, scores):
    print(f'  {coco_classes[lbl]:>10}  {sc:.1%}')

colors = plt.cm.tab10(np.linspace(0, 1, 10))
inst_label_rgb = np.full_like(photo_rgb, 255)
for i in range(len(labels)):
    color = (np.array(colors[i][:3]) * 255).astype(np.uint8)
    inst_label_rgb[masks[i] > 0.5] = color

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(photo_rgb); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(inst_label_rgb); axes[1].set_title('Mask R-CNN per-instance masks'); axes[1].axis('off')
legend_handles = [mpatches.Patch(color=colors[i], label=f'{coco_classes[labels[i]]} #{i+1} ({scores[i]:.0%})')
                  for i in range(len(labels))]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: NASA (phạm vi công cộng), qua <a href="https://commons.wikimedia.org/wiki/File:Apollo_11_Crew.jpg" target="_blank" rel="noopener">Wikimedia Commons</a></p>

Ba phi hành gia đứng vai kề vai, chạm nhau dọc theo nhiều ranh giới cánh tay và thân người, và thuộc chính xác cùng một lớp COCO (`person`) — đúng trường hợp mà một mặt nạ ngữ nghĩa sẽ gộp họ thành một khối không phân biệt. Mask R-CNN vẫn tách họ gọn gàng thành ba thực thể riêng biệt, vì bộ phát hiện của nó đề xuất ba hộp riêng biệt trước (mỗi phi hành gia một hộp) và phần đầu mặt nạ chỉ nhìn vào bên trong mỗi lần một hộp, y như phần đầu mặt nạ ở trên. Một phát hiện thứ tư, giả, cũng xuất hiện: một mặt nạ `clock` nhỏ với độ tin cậy 84%, trên cổ tay của phi hành gia bên trái — tấm danh sách kiểm tra tròn, có gương gắn trên cổ tay áo của Armstrong, bị đọc nhầm thành mặt đồng hồ.

### Bài tập

1. Giảm `dist` trong `make_scene` từ khoảng `(6, 9)` xuống `(2, 5)`, làm các hình tròn chồng lấn nhiều hơn hẳn (tâm gần nhau hơn). Độ chính xác đếm thực thể của phương pháp bỏ phiếu độ lệch có giữ vững, hay nó cũng bắt đầu thất bại — và nếu có, thất bại trông như thế nào (thực thể bị gộp, thực thể bị tách, hay thứ gì khác)?
2. Phép phân cụm ở đây dùng `peak_dist=4` cho triệt tiêu điểm không cực đại trên bộ tích lũy phiếu. Hãy thử `peak_dist=8`. Độ chính xác có cải thiện, tệ đi, hay trở nên nhạy cảm với việc hai tâm hình tròn thực tình cờ gần nhau đến đâu trong một cảnh nhất định?
3. Bài này chỉ luôn có 2 hình tròn mỗi cảnh. Mở rộng `make_scene` để đặt một số lượng hình tròn ngẫu nhiên (1 đến 4) và điều chỉnh cận vòng lặp của `cluster_instances` tương ứng. Độ chính xác đếm thực thể của cách tiếp cận bỏ phiếu độ lệch có giữ vững khi số thực thể thực tăng lên không?
4. Phần đầu mặt nạ ở trên được huấn luyện và đánh giá dùng tâm hộp *chuẩn*, né tránh hoàn toàn việc phát hiện. Hãy nhiễu loạn tâm của mỗi vùng cắt vài điểm ảnh ngẫu nhiên trước khi cắt (mô phỏng một bộ phát hiện không hoàn hảo) và chạy lại. IoU mặt nạ suy giảm bao nhiêu khi hộp được căn giữa kém chính xác hơn — và điều đó có khớp với sự phụ thuộc thực tế của Mask R-CNN vào chất lượng bộ phát hiện không?